# Lucia v1.1: the text-in-pictures lane

The lane of the v1.1 text work (the reads of shown text in pictures, then the continuation): the baseline read extracts each released v1.1 adapter and its base on held-out COCO-Text pictures (COCO 2017 photographs whose text is legible) with the trained closer-look ask, at LoRA strengths 1 and 0.5 (the extraction's adapter_scale), and ships the rows to the private twin, where the read counts the pictures whose shown words the row names.

Runtime: a GPU (the RTX PRO 6000) and the `HF_TOKEN` secret with read access to the accepted rows and write access to the private twin `AbstractPhil/qwen-symbolic-thinking-image-training-private`, where every file of these runs lives. Run C1, restart the session (Runtime > Restart session), then run C2. Each run leaves `runs/done/<name>.json` in the twin; the queue cell ends after 45 idle minutes.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.65", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through this line's queue (runs/queue_lucia_v11_9b.json in the private twin; new runs can be added while it runs)
import os
os.environ["QST_REPO"] = "AbstractPhil/qwen-symbolic-thinking-image-training-private"   # before the package is imported: every file of these runs stays private
LANE = "lucia_v11_text"

from qwen_symbolic_thinking import REPO, __version__
from qwen_symbolic_thinking.queue import run_queue

print("package", __version__, "| project repo", REPO, "| lane", LANE)
ran = run_queue(LANE, wait_hours=12)
print("ran:", ran)
